In [ ]:
# ============================================================
# ENDES 2024 - INTEGRACIÓN DE MICRODATOS
# Región Puno - Niños de 6 a 35 meses
# ============================================================

import pandas as pd
import numpy as np

print("Proyecto: ENDES 2024 - Anemia infantil en Puno")
print("Etapa: Integración inicial de módulos")

In [ ]:
# ============================================================
# 1. CARGA DE MÓDULOS ENDES 2024
# ============================================================

import pandas as pd
import numpy as np

rech0 = pd.read_csv(
    "RECH0_2024.csv",
    sep=";",
    encoding="latin-1",
    low_memory=False
)

rech6 = pd.read_csv(
    "RECH6_2024.csv",
    sep=",",
    encoding="latin-1",
    low_memory=False
)

print("RECH0:", rech0.shape)
print("RECH6:", rech6.shape)

print("\nPrimeras columnas RECH0:")
print(rech0.columns[:10].tolist())

print("\nPrimeras columnas RECH6:")
print(rech6.columns[:10].tolist())

In [ ]:
# ============================================================
# CORRECCIÓN DE CARGA RECH0 - ENDES 2024
# ============================================================

rech0 = pd.read_csv(
    "RECH0_2024.csv",
    sep=",",
    encoding="latin-1",
    low_memory=False
)

print("RECH0:", rech0.shape)

print("\nPrimeras columnas RECH0:")
print(rech0.columns[:10].tolist())

In [ ]:
# ============================================================
# LIMPIEZA DE NOMBRES DE COLUMNAS
# ============================================================

rech0.columns = (
    rech0.columns
    .str.replace("\ufeff", "", regex=False)
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
)

rech6.columns = (
    rech6.columns
    .str.replace("\ufeff", "", regex=False)
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
)

print("Primeras columnas RECH0:")
print(rech0.columns[:10].tolist())

print("\nPrimeras columnas RECH6:")
print(rech6.columns[:10].tolist())

In [ ]:
# ============================================================
# VERIFICACIÓN DEL AÑO 2024
# ============================================================

print("Años encontrados en RECH0:")
print(rech0["ID1"].value_counts(dropna=False))

print("\nAños encontrados en RECH6:")
print(rech6["ID1"].value_counts(dropna=False))

Vamos a comprobar que HHID exista en ambos módulos y que en RECH0 no haya hogares duplicados. Esto es necesario para hacer la integración de forma segura.

In [ ]:
# ============================================================
# 4. VERIFICACIÓN DE LA LLAVE HHID - ENDES 2024
# ============================================================

print("¿HHID existe en RECH0?", "HHID" in rech0.columns)
print("¿HHID existe en RECH6?", "HHID" in rech6.columns)

print("\nCantidad de registros:")
print("RECH0:", len(rech0))
print("RECH6:", len(rech6))

print("\nCantidad de HHID únicos:")
print("RECH0:", rech0["HHID"].nunique())
print("RECH6:", rech6["HHID"].nunique())

print("\nDuplicados de HHID en RECH0:")
print(rech0["HHID"].duplicated().sum())

Integración RECH6 + RECH0

Se integra la información de los niños con la información de sus hogares utilizando HHID como llave común.

In [ ]:
# ============================================================
# 5. INTEGRACIÓN DE RECH6 + RECH0 - ENDES 2024
# ============================================================

endes_2024 = pd.merge(
    rech6,
    rech0,
    on="HHID",
    how="left",
    validate="many_to_one",
    suffixes=("_RECH6", "_RECH0")
)

print("Integración realizada correctamente.")

print("\nDimensión RECH6 antes de integrar:")
print(rech6.shape)

print("\nDimensión de la base integrada:")
print(endes_2024.shape)

In [ ]:
# ============================================================
# 5.1 CONTROL DE COINCIDENCIAS
# ============================================================

sin_coincidencia = endes_2024["HV024"].isna().sum()

print("Registros sin coincidencia con RECH0:")
print(sin_coincidencia)

In [ ]:
# ============================================================
# 5.2 VALIDACIÓN DE LA INTEGRACIÓN
# ============================================================

print("Registros antes de integrar:", len(rech6))
print("Registros después de integrar:", len(endes_2024))

if len(rech6) == len(endes_2024):
    print("✅ No se perdieron registros durante la integración.")
else:
    print("⚠️ Revisar la integración.")

Se verificó que el módulo RECH0 presenta un único registro por hogar, permitiendo integrar RECH6 y RECH0 mediante una relación muchos-a-uno utilizando HHID como identificador común. Posteriormente, se comprobó la conservación de los registros infantiles y la correspondencia con la información del hogar.

Ahora que ya tenemos RECH0 y RECH6 integrados, seleccionamos únicamente los registros correspondientes a la región Puno usando la variable HV024.

In [ ]:
# ============================================================
# 6.1 VERIFICACIÓN DE DEPARTAMENTO - ENDES 2024
# ============================================================

print("Códigos encontrados en HV024:")

print(
    endes_2024["HV024"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# ============================================================
# 6.2 CONTEO DE REGISTROS DE PUNO
# ============================================================

registros_puno_2024 = (endes_2024["HV024"] == 21).sum()

print("Registros correspondientes a Puno:")
print(registros_puno_2024)

In [ ]:
# ============================================================
# 6.3 FILTRO REGIÓN PUNO
# ============================================================

puno_2024 = endes_2024[
    endes_2024["HV024"] == 21
].copy()

print("Base filtrada para Puno:")
print(puno_2024.shape)

In [ ]:
# ============================================================
# 6.4 VALIDACIÓN DEL FILTRO
# ============================================================

print("Valores de HV024 después del filtro:")
print(puno_2024["HV024"].unique())

if puno_2024["HV024"].nunique() == 1 and puno_2024["HV024"].iloc[0] == 21:
    print("✅ La base contiene únicamente registros de Puno.")
else:
    print("⚠️ Revisar el filtro de departamento.")

In [ ]:
# ============================================================
# 7.1 FILTRO DE EDAD: 6 A 35 MESES - ENDES 2024
# ============================================================

puno_6_35_2024 = puno_2024[
    (puno_2024["HC1"] >= 6) &
    (puno_2024["HC1"] <= 35)
].copy()

print("Niños de 6 a 35 meses en Puno:")
print(puno_6_35_2024.shape)

In [ ]:
# ============================================================
# 7.2 VALIDACIÓN DEL RANGO DE EDAD
# ============================================================

print("Edad mínima:", puno_6_35_2024["HC1"].min(), "meses")
print("Edad máxima:", puno_6_35_2024["HC1"].max(), "meses")

print("\nCantidad de registros:")
print(len(puno_6_35_2024))

In [ ]:
# ============================================================
# 7.3 COMPROBACIÓN AUTOMÁTICA
# ============================================================

if (
    puno_6_35_2024["HC1"].min() >= 6
    and puno_6_35_2024["HC1"].max() <= 35
):
    print("✅ Todos los registros están entre 6 y 35 meses.")
else:
    print("⚠️ Revisar el filtro de edad.")

In [ ]:
# ============================================================
# 8.1 REVISIÓN DE LA VARIABLE HC57 - ENDES 2024
# ============================================================

print("Valores encontrados en HC57:")

print(
    puno_6_35_2024["HC57"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# ============================================================
# 8.2 DISTRIBUCIÓN PORCENTUAL DE HC57
# ============================================================

porcentaje_hc57_2024 = (
    puno_6_35_2024["HC57"]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
    * 100
).round(2)

print("Porcentaje por categoría de HC57:")
print(porcentaje_hc57_2024)

In [ ]:
# ============================================================
# 9.1 DEPURACIÓN DE HC57 - ENDES 2024
# ============================================================

datos_2024 = puno_6_35_2024[
    puno_6_35_2024["HC57"].isin([1, 2, 3, 4])
].copy()

print("Registros antes de depurar:", len(puno_6_35_2024))
print("Registros después de depurar:", len(datos_2024))

print("\nRegistros eliminados:")
print(len(puno_6_35_2024) - len(datos_2024))

In [ ]:
# ============================================================
# 9.2 CREACIÓN DE LA VARIABLE OBJETIVO
# ============================================================

datos_2024["target_anemia"] = np.where(
    datos_2024["HC57"].isin([1, 2, 3]),
    1,
    0
)

print("Variable target_anemia creada correctamente.")

In [ ]:
# ============================================================
# 9.3 DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================================

print(
    datos_2024["target_anemia"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# 9.4 TABLA CON ETIQUETAS Y PORCENTAJES
# ============================================================

tabla_anemia_2024 = (
    datos_2024["target_anemia"]
    .value_counts()
    .rename(index={
        1: "Con anemia",
        0: "Sin anemia"
    })
    .to_frame("Frecuencia")
)

tabla_anemia_2024["Porcentaje"] = (
    tabla_anemia_2024["Frecuencia"]
    / len(datos_2024)
    * 100
).round(2)

print(tabla_anemia_2024)

In [ ]:
# ============================================================
# 10.1 DISTRIBUCIÓN FINAL DE ANEMIA - ENDES 2024
# ============================================================

tabla_final_2024 = (
    datos_2024["target_anemia"]
    .value_counts()
    .rename(index={
        1: "Con anemia",
        0: "Sin anemia"
    })
    .to_frame("Frecuencia")
)

tabla_final_2024["Porcentaje"] = (
    tabla_final_2024["Frecuencia"]
    / len(datos_2024)
    * 100
).round(2)

print("Distribución final de anemia - ENDES 2024")
print(tabla_final_2024)

In [ ]:
# ============================================================
# 10.2 VALIDACIÓN FINAL
# ============================================================

print("Total de registros finales:", len(datos_2024))
print("Valores de target_anemia:", datos_2024["target_anemia"].unique())

print("\nValores faltantes en target_anemia:")
print(datos_2024["target_anemia"].isna().sum())

In [ ]:
# ============================================================
# EXPORTAR ENDES 2024 DIRECTAMENTE A EXCEL (.xlsx)
# ============================================================

nombre_excel = "ENDES_PUNO_ANEMIA_2024.xlsx"

datos_2024.to_excel(
    nombre_excel,
    index=False
)

print("✅ Archivo Excel generado correctamente:")
print(nombre_excel)